This notebook references Lecture 8 from the Deep Learning course by Proffessor Bryce.

This notebook implements backpropagation on a simple feed forward neural network. 

Regression Dataset -> https://archive.ics.uci.edu/dataset/360/air+quality \
Classification Dataset -> https://archive.ics.uci.edu/dataset/728/toxicity-2 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Sample Synthetic data

In [2]:
# Synthetic data for regression and classification

def synthetic_regression_data(N, # number of datapoints to generate
                              n_input, # number of input variables 
                              ): 
    
    # 1. determine weights and bias randomly for the data
    W = np.random.randn(n_input)
    b = np.random.randn()
    
    # 2. Generate data
    X = np.random.randint(10, size=(N, n_input))
    y = X.dot(W) + b + np.random.rand(N)
    
    
    # 4. return the W, b and data
    return X, y
    
    
def synthetic_classification_data(N, n_input):
    
    # 1. determine weights and bias randomly for the data
    W = np.random.randn(n_input)
    b = np.random.randn()
    
    # 2. Generate data
    X = np.random.randint(10, size=(N, n_input))
    y_ = X.dot(W) + b + np.random.rand()
    threshold = np.random.random()
    y = (y_ >= threshold).astype(int)
    
    
    # 4. return the W, b and data
    return threshold, X, y
    

# Working of Back Propagation



In [46]:
class Neuron():
    
    def __init__(self, n_features, type="linear"):
        self.W = np.random.rand(n_features, 1)
        print(self.W.shape)
        self.b = np.random.rand()
        self.type = type
        self.W_grad = None
        self.b_grad = None
    
    def forward(self, X): # this function is used during the training process of the model
        self.X = X
        z = X.dot(self.W) + self.b

        if self.type == "linear":
            a = self._linear(z)
            self.derivative = np.ones_like(a)
            return a
        elif self.type == "sigmoid":
            a = self._sigmoid(z)
            self.derivative = self._sigmoid_derivative(a)
            return a
        elif self.type == "tanh":
            a = self._tanh(z)
            self.derivative = self._tanh_derivative(a)
            return a
        elif self.type == "relu":
            a = self._relu(z)
            self.derivative = self._relu_derivative(a)
            return a
        
    def backward(self, dl_da = None):
        if dl_da is not None:
            delta = dl_da * self.derivative
            self.W_grad = self.X.T @ delta / self.X.shape[0]
            self.b_grad = np.sum(delta)/ self.X.shape[0]
            return delta @ self.W.T
        
        return 0
    
    def update(self, alpha):
        self.W -= (alpha * self.W_grad)
        self.b -= (alpha * self.b_grad)
    
    def predict(self, X): # this function is used for making the inferences from the trained model
        y = X.dot(self.W) + self.b
        
        if self.type == "linear":
            return y
        elif self.type == "sigmoid":
            return self._sigmoid(y)
        elif self.type == "tanh":
            return self._tanh(y)
        elif self.type == "reLu":
            return self._relu(y)
            
    def _linear(self, z):
        return z
    
    def _sigmoid(self, z):
        return 1/ (1 + np.exp(-z))
    
    def _tanh(self, z):
        return np.tanh(z)
    
    def _relu(self, z):
        return np.maximum(0, z)
    
    def _sigmoid_derivative(self, a):
        return a * (1-a)
    
    def _tanh_derivative(self, a):
        return 1 - (a**2)
    
    def _relu_derivative(self, a):
        return (a>0).astype(int)
    
    
class Layer():
    
    def __init__(self, size = (None, None, None), activation = None):
        N, n_features, n_neurons = size
        
        self.size = size
        self.activation = activation
        self.neurons = {}
        
        for i in range(n_neurons):
            self.neurons[f'neuron_{i+1}'] = Neuron(n_features, self.activation)
            
    def forward(self, X):
        outputs = {}
        i=0
        for neuron_name, neuron in self.neurons.items():
            
            outputs[neuron_name] = np.array(neuron.forward(X)).reshape(-1, 1)
            #print(outputs[neuron_name].shape)
            
        return np.hstack([*outputs.values()])
    
    def predict(self, X):
        outputs = {}
        i=0
        for neuron_name, neuron in self.neurons.items():
            
            outputs[neuron_name] = np.array(neuron.predict(X)).reshape(-1, 1)
            #print(outputs[neuron_name].shape)
            
        return np.hstack([*outputs.values()])
    
    def backward(self, dl_da = None):
        dl_dx = 0
        for i, neuron in enumerate(self.neurons.values()):
            dl_dx += neuron.backward(dl_da[:, i:i+1])
        return dl_dx
    
    def update(self, alpha):
        for _, neuron in self.neurons.items():
            neuron.update(alpha)
            
            
class RegressionModel:
    
    def __init__(self, input_size):
        
        N, _ = input_size
        
        self.layer1 = Layer((*input_size, 3), activation="tanh")
        self.layer2 = Layer((N, 3, 2), activation="tanh")
        self.output_layer = Layer((N, 2, 1), activation="linear")
        
    def forward(self, X):
        y = self.layer1.forward(X)
        #print(y.shape)
        y = self.layer2.forward(y)
        #print(y.shape)
        y = self.output_layer.forward(y)
        #print(yo.shape)
        return y
        
    def predict(self, X):
        y = self.layer1.predict(X)
        #print(y.shape)
        y = self.layer2.predict(y)
        #print(y.shape)
        y = self.output_layer.predict(y)
        #print(yo.shape)
        
        return y
    
    def backward(self, y, y_):
        dl_da = y_ - y
        dl_dz_output = self.output_layer.backward(dl_da)
        dl_dz_layer2 = self.layer2.backward(dl_da=dl_dz_output)
        self.layer1.backward(dl_da=dl_dz_layer2)
    
    def update(self, alpha):
        self.layer1.update(alpha)
        self.layer2.update(alpha)
        self.output_layer.update(alpha)
        
        
    def fit(self, X, y, alpha = 0.01, epochs = 100):
        
        m, n = X.shape
        self.alpha = alpha
        
        for epoch in range(epochs):
            
            y_ = self.forward(X)
            self.backward(y, y_)
            self.update(alpha)

            #print(f"Epoch {epoch+1} => ", np.mean(error ** 2))
            
            
class ClassificationModel:
    
    def __init__(self, input_size, threshold):
        
        N, _ = input_size
        self.threshold = threshold
        self.layer1 = Layer((*input_size, 3), activation="tanh")
        print(self.layer1.size)
        self.layer2 = Layer((N, 3, 2), activation="tanh")
        self.output_layer = Layer((N, 2, 1), activation="sigmoid")
        
    def forward(self, X):
        y = self.layer1.forward(X)
        #print(y.shape)
        y = self.layer2.forward(y)
        #print(y.shape)
        yo = self.output_layer.forward(y)
        #print(yo.shape)
        
        return yo 

    def predict(self, X):
        y = self.layer1.predict(X)
        print(y.shape)
        y = self.layer2.predict(y)
        print(y.shape)
        y = self.output_layer.predict(y)
        print(y.shape)
        
        return (y >= self.threshold).astype(int)
    
    def backward(self, y, y_):
        dl_da = y_ - y
        dl_dz_output = self.output_layer.backward(dl_da)
        dl_dz_layer2 = self.layer2.backward(dl_da=dl_dz_output)
        self.layer1.backward(dl_da=dl_dz_layer2)
    
    def update(self, alpha):
        self.layer1.update(alpha)
        self.layer2.update(alpha)
        self.output_layer.update(alpha)
        
    def fit(self, X, y, alpha = 0.01, epochs = 100):
        
        m, n = X.shape
        self.alpha = alpha
        
        for epoch in range(epochs):
            
            y_ = self.forward(X)
            self.backward(y, y_)
            self.update(alpha)
            #print(epoch)
            #print(f"Epoch {epoch+1} => ", np.mean(error ** 2))
                    
                    
    

Testing on synthetic regression data

In [47]:
X_reg, y_reg = synthetic_regression_data(100, 5)

reg_model = RegressionModel((100, 5))

reg_model.fit(X_reg, y_reg)
print("Fit complete")

y_pred = reg_model.predict(X_reg)

(5, 1)
(5, 1)
(5, 1)
(3, 1)
(3, 1)
(2, 1)
Fit complete


Testing on synthetic classification data

In [48]:
cls_threshold, X_cls, y_cls = synthetic_classification_data(100, 5)

cls_model = ClassificationModel((100, 5), 0.5)

cls_model.fit(X_cls, y_cls)
print("Fit complete")

y_pred = cls_model.predict(X_cls)

(5, 1)
(5, 1)
(5, 1)
(100, 5, 3)
(3, 1)
(3, 1)
(2, 1)
Fit complete
(100, 3)
(100, 2)
(100, 1)


Testing on real datasets mentioned earlier

In [31]:
from ucimlrepo import fetch_ucirepo 
  
# fetching classification dataset 
toxicity = fetch_ucirepo(id=728) 
  
# data (as pandas dataframes) 
X_toxic = toxicity.data.features 
y_toxic = toxicity.data.targets 


In [32]:
# for simplicity sake we are skipping all the categorical features and considering only around 10 random features

toxicity_features = ['MATS3v', 'MATS3s', 'MATS3p', 'nHBDon_Lipinski', 'minHBint8', 'MATS3e', 'MATS3c', 'MATS3m'] 
X_toxic = X_toxic[toxicity_features]

y_toxic = (np.array(y_toxic) != "NonToxic").astype(int)

In [33]:
from sklearn.metrics import accuracy_score as accsc, mean_squared_error as mse, r2_score as r2, precision_score as presc

In [38]:
X_toxic.shape

(171, 8)

In [41]:
cls_model_real = ClassificationModel(X_toxic.shape, 0.6)
cls_model_real.fit(np.array(X_toxic), np.asarray(y_toxic).reshape(-1, ))
y_cls_model_real = cls_model_real.predict(np.array(X_toxic))

print("Accuracy score : ", accsc(y_toxic, y_cls_model_real))
print("Precision score : ", presc(y_toxic, y_cls_model_real))


(171, 8, 3)
(171, 3)
(171, 2)
(171, 1)
Accuracy score :  0.672514619883041
Precision score :  0.0


c:\random\Desktop\Deep Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [37]:
y_prob = cls_model.forward(np.array(X_toxic))
print(y_prob.min(), y_prob.max())

ValueError: shapes (171,8) and (5,1) not aligned: 8 (dim 1) != 5 (dim 0)

In [50]:
# fetching classification dataset https://archive.ics.uci.edu/dataset/265/physicochemical+properties+of+protein+tertiary+structure

reg_data = pd.read_csv('./datasets/CASP.csv')
reg_data.info()

X_casp, y_casp = reg_data.drop(['RMSD'], axis=1), reg_data['RMSD']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45730 entries, 0 to 45729
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RMSD    45730 non-null  float64
 1   F1      45730 non-null  float64
 2   F2      45730 non-null  float64
 3   F3      45730 non-null  float64
 4   F4      45730 non-null  float64
 5   F5      45730 non-null  float64
 6   F6      45730 non-null  float64
 7   F7      45730 non-null  float64
 8   F8      45730 non-null  int64  
 9   F9      45730 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 3.5 MB


In [ ]:
reg_model_real = RegressionModel(X_casp.shape)
reg_model_real.fit(X_casp, np.asarray(y_casp), epochs=100)
_, y_reg_model_real = reg_model_real.predict(X_casp)

print("R2 score : ", r2(y_casp, y_reg_model_real))
print("MSE : ", mse(y_casp, y_reg_model_real))


(9, 1)
(9, 1)
(9, 1)
(3, 1)
(3, 1)
(2, 1)


c:\random\Desktop\Deep Learning\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:81: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
c:\random\Desktop\Deep Learning\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:81: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
c:\random\Desktop\Deep Learning\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:81: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  r

Important learning here:

the version implemented here is very naive, and is at the neuron abstraction level. 

A better, optimized version is written in the next notebook.